## 01 — Feature Engineering

> **Costruzione del dataset di features per il modello NBA win prediction.**
> Input: `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`
> Output: `data/3_features/features_nba_data_2000-01_2025-26.csv`

---

**Pipeline**
1. Setup & caricamento dati
2. Feature Plan — shortlist dall'EDA Bivariata + mappa colonne
3. Vista per-team — una riga per (team_id, game_id), ordinata cronologicamente
4. Rolling features — medie mobili su N partite precedenti con shift(1)
5. Context features — streak, rest_days, is_back_to_back
6. Ratio features — fg_pct, fg3_pct, ft_pct
7. Normalizzazioni — z-score within-season (ts_pct, pace) + season z-norm
8. Ricostruzione vista partita — join home / away + win_rate_prev_season
9. Feature differenziali — home_X − away_X + definizione target
10. Flag stagione anomala — is_covid_bubble (2019-20)
11. Filtering — MIN_GAMES_REQUIRED
12. Train / Val / Test split temporale
13. Salvataggio

### 1. Setup

In [ ]:
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent.parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import polars as pl

from src.loader import load_config

config = load_config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

NOTEBOOK_VERSION = "1.0.0"
log.info(f"Feature Engineering notebook v{NOTEBOOK_VERSION} — initialized")

In [ ]:
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
FEATURES_DIR  = ROOT / "data" / "3_features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

IN_FILE  = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"
OUT_FILE = FEATURES_DIR  / f"features_nba_data_{DATASET_VERSION}.csv"

WINDOW_N           = 10
MIN_GAMES_REQUIRED = 5

log.info(f"Input             : {IN_FILE}")
log.info(f"Output            : {OUT_FILE}")
log.info(f"Rolling window N  : {WINDOW_N}")
log.info(f"Min games required: {MIN_GAMES_REQUIRED}")

In [ ]:
df = pl.read_csv(IN_FILE, try_parse_dates=True)
log.info(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
log.info(f"Columns: {df.columns}")
df.head(3)

### 2. Feature Plan

#### 2.1 Shortlist dall'EDA Bivariata (`02_eda_bivariate.ipynb` — sezione 8.4)

```
╔══════════════════════════════════════════════════════════════════════════════╗
║  INCLUDI DIRETTAMENTE                                                        ║
║    • net_rating      — predittore più forte, bassa ridondanza con ts_pct     ║
║    • pts             — forte segnale, interpretabile                          ║
║    • ts_pct          — efficienza sintetica, trend temporale → z-score       ║
║    • pace            — strutturale, trend temporale → z-score within-season  ║
║    • home_away       — home advantage significativo (~60% win rate home)      ║
║                                                                              ║
║  INCLUDI CON TRASFORMAZIONE                                                  ║
║    • fg_pct   = fgm / fga   (ratio al posto delle raw)                       ║
║    • fg3_pct  = fg3m / fg3a                                                  ║
║    • ft_pct   = ftm / fta                                                    ║
║    • season (z-norm)  — per catturare drift strutturale di regole NBA         ║
║                                                                              ║
║  INCLUDI COME FEATURE LAGGED (Feature Engineering)                           ║
║    • net_rating_rolling_N   — media mobile N partite precedenti              ║
║    • win_rate_prev_season   — stabilità inter-stagionale                      ║
║                                                                              ║
║  ESCLUDI                                                                     ║
║    • plus_minus   — data leakage (determinato dal risultato)                  ║
║    • off_rating   — ridondante se net_rating incluso                          ║
║    • def_rating   — ridondante se net_rating incluso                          ║
║    • fgm, fga, fg3m, fg3a, ftm, fta  — sostituiti dai ratio                  ║
║    • game_id, team_id   — identificatori tecnici                              ║
║    • is_outlier_flag    — flag qualità, non predittore                        ║
║                                                                              ║
║  STAGIONI ANOMALE DA GESTIRE                                                  ║
║    • 2019-20 (COVID bubble) — testare esclusione o dummy flag                 ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

---

#### 2.2 Mappa colonne: raw → trasformazione → feature finale

| Colonna raw | Trasformazione | Feature finale | Note |
|-------------|----------------|----------------|------|
| `net_rating` | `rolling_mean(N)`, `shift(1)` per team | `net_rating_rolling_N` | Più forte predittore singolo (EDA) |
| `pts` | `rolling_mean(N)`, `shift(1)` per team | `avg_pts_last_N` | Trend offensivo recente |
| `ts_pct` | `rolling_mean(N)`, `shift(1)` per team | `avg_ts_pct_last_N` | Efficienza tiro negli ultimi N game |
| `pace` | `rolling_mean(N)`, `shift(1)` per team | `avg_pace_last_N` | Ritmo di gioco recente |
| `wl` | win indicator + `rolling_mean(N)`, `shift(1)` | `winrate_last_N` | Forma recente (win rate ultimi N) |
| `wl` | conteggio streak consecutivi, `shift(1)` | `streak` | Momentum (int con segno: +win, −loss) |
| `game_date` | `diff()` per team | `rest_days` | Giorni di riposo dal game precedente |
| `rest_days` | `== 1` | `is_back_to_back` | Flag back-to-back (bool) |
| `fgm / fga` | divisione; `fga == 0 → null` | `fg_pct` | Efficienza tiro dal campo |
| `fg3m / fg3a` | divisione; `fg3a == 0 → null` | `fg3_pct` | Efficienza da tre punti |
| `ftm / fta` | divisione; `fta == 0 → null` | `ft_pct` | Efficienza ai liberi |
| `ts_pct` | z-score within-season | `ts_pct_zscore` | Normalizzazione drift strutturale |
| `pace` | z-score within-season | `pace_zscore` | Normalizzazione drift strutturale |
| `season` | indice ordinale z-norm | `season_zscore` | Modellazione tendenza storica |
| `wl` (stagione prec.) | media vittorie per (team, season−1) | `win_rate_prev_season` | Stabilità inter-stagionale |

### 3. Vista per-team

Reshape: una riga per (team_id, game_id) con colonne uniformi.
Ordinamento cronologico per `team_id + game_date` — prerequisito per tutti i rolling.

In [ ]:
KEEP_COLS = [
    "game_id", "game_date", "season", "team_id", "team_abbreviation",
    "home_away", "wl", "pts", "plus_minus",
    "net_rating", "ts_pct", "pace",
    "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
    "is_outlier_flag",
]

team_view = df.select(KEEP_COLS).sort(["team_id", "game_date"])

log.info(f"Vista per-team: {team_view.shape[0]:,} rows × {team_view.shape[1]} columns")
log.info(f"Teams: {team_view['team_id'].n_unique()} | Seasons: {team_view['season'].n_unique()}")
log.info(f"Date range: {team_view['game_date'].min()} → {team_view['game_date'].max()}")
team_view.head(3)

### 4. Rolling features (shift(1) — no data leakage)

Ogni media mobile usa `shift(1)` dentro `.over("team_id")`, in modo che la feature
alla riga *i* rifletta esclusivamente le *N* partite **precedenti** a quella corrente.
`min_periods=MIN_GAMES_REQUIRED` forza `null` quando la storia disponibile è inferiore
alla soglia minima — le righe corrispondenti verranno scartate al passo 11.

In [ ]:
team_view = team_view.with_columns([
    pl.col("net_rating")
        .rolling_mean(window_size=WINDOW_N, min_periods=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"net_rating_rolling_{WINDOW_N}"),

    pl.col("pts").cast(pl.Float64)
        .rolling_mean(window_size=WINDOW_N, min_periods=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_pts_last_{WINDOW_N}"),

    pl.col("ts_pct")
        .rolling_mean(window_size=WINDOW_N, min_periods=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_ts_pct_last_{WINDOW_N}"),

    pl.col("pace")
        .rolling_mean(window_size=WINDOW_N, min_periods=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_pace_last_{WINDOW_N}"),

    (pl.col("wl") == "W").cast(pl.Float64)
        .rolling_mean(window_size=WINDOW_N, min_periods=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"winrate_last_{WINDOW_N}"),
])

rolling_cols = [
    f"net_rating_rolling_{WINDOW_N}", f"avg_pts_last_{WINDOW_N}",
    f"avg_ts_pct_last_{WINDOW_N}", f"avg_pace_last_{WINDOW_N}",
    f"winrate_last_{WINDOW_N}",
]
for c in rolling_cols:
    log.info(f"{c}: null={team_view[c].null_count():,}")

### 5. Context features

In [ ]:
def _streak_per_team(wl_vals: np.ndarray) -> np.ndarray:
    """Signed streak at each position using only PREVIOUS games (no leakage).

    Positive = consecutive wins ending before this game.
    Negative = consecutive losses ending before this game.
    Position 0 → streak = 0 (no prior history).
    """
    n = len(wl_vals)
    streaks = np.zeros(n, dtype=np.int32)
    for i in range(1, n):
        prev = wl_vals[i - 1]
        count = 1
        j = i - 2
        while j >= 0 and wl_vals[j] == prev:
            count += 1
            j -= 1
        streaks[i] = int(count) if prev == "W" else -int(count)
    return streaks


# Compute per-team streak in pandas (concise; ~30 teams × ~2 000 games each)
_key = ["game_id", "team_id"]
_tmp = (
    team_view.select(_key + ["game_date", "wl"])
    .to_pandas()
    .sort_values(["team_id", "game_date"])
    .reset_index(drop=True)
)
_tmp["streak"] = 0
for _, grp in _tmp.groupby("team_id", sort=True):
    _tmp.loc[grp.index, "streak"] = _streak_per_team(grp["wl"].values)

streak_pl = (
    pl.from_pandas(_tmp[_key + ["streak"]])
    .with_columns(pl.col("team_id").cast(team_view["team_id"].dtype))
)
team_view = team_view.join(streak_pl, on=_key, how="left")

log.info(
    f"streak: min={team_view['streak'].min()}, max={team_view['streak'].max()}, "
    f"null={team_view['streak'].null_count()}"
)

In [ ]:
team_view = (
    team_view
    .with_columns([
        (
            pl.col("game_date")
            - pl.col("game_date").shift(1).over("team_id")
        ).dt.total_days().alias("rest_days")
    ])
    .with_columns([
        (pl.col("rest_days") == 1).fill_null(False).alias("is_back_to_back")
    ])
)

b2b_n = int(team_view.filter(pl.col("is_back_to_back"))["game_id"].len())
log.info(f"rest_days      : null={team_view['rest_days'].null_count():,} (prima partita per team)")
log.info(f"is_back_to_back: {b2b_n:,} partite ({b2b_n / team_view.height:.1%})")

### 6. Ratio features

Calcolati dal dato pulito (non dal rolling).
Il denominatore `== 0` produce `null` — non `inf` — per evitare di distorcere
le statistiche downstream e i modelli gradient-boosted.

In [ ]:
team_view = team_view.with_columns([
    pl.when(pl.col("fga") > 0)
      .then(pl.col("fgm").cast(pl.Float64) / pl.col("fga").cast(pl.Float64))
      .otherwise(None)
      .alias("fg_pct"),

    pl.when(pl.col("fg3a") > 0)
      .then(pl.col("fg3m").cast(pl.Float64) / pl.col("fg3a").cast(pl.Float64))
      .otherwise(None)
      .alias("fg3_pct"),

    pl.when(pl.col("fta") > 0)
      .then(pl.col("ftm").cast(pl.Float64) / pl.col("fta").cast(pl.Float64))
      .otherwise(None)
      .alias("ft_pct"),
])

for col in ["fg_pct", "fg3_pct", "ft_pct"]:
    null_n = team_view[col].null_count()
    log.info(f"{col}: null={null_n} (zero-denominator games → null, not inf)")

### 7. Normalizzazioni

- **`ts_pct_zscore`** e **`pace_zscore`**: z-score *within-season* per catturare il drift
  strutturale dell'era NBA. I valori assoluti di `pace` e `ts_pct` sono aumentati
  significativamente negli ultimi 25 anni; confrontarli direttamente introduce un bias
  sistematico nel modello.
- **`season_zscore`**: z-norm sull'indice stagionale numerico per permettere al modello
  di apprendere trend longitudinali (esplosione del tiro da tre, pace crescente).

In [ ]:
season_stats = (
    team_view
    .group_by("season")
    .agg([
        pl.col("ts_pct").mean().alias("ts_pct_mean"),
        pl.col("ts_pct").std(ddof=1).alias("ts_pct_std"),
        pl.col("pace").mean().alias("pace_mean"),
        pl.col("pace").std(ddof=1).alias("pace_std"),
    ])
)

team_view = (
    team_view
    .join(season_stats, on="season", how="left")
    .with_columns([
        ((pl.col("ts_pct") - pl.col("ts_pct_mean")) / pl.col("ts_pct_std"))
            .alias("ts_pct_zscore"),
        ((pl.col("pace") - pl.col("pace_mean")) / pl.col("pace_std"))
            .alias("pace_zscore"),
    ])
    .drop(["ts_pct_mean", "ts_pct_std", "pace_mean", "pace_std"])
)

# Season numeric index: extract year from "2000-01" -> 2000, then z-norm
team_view = team_view.with_columns(
    pl.col("season").str.slice(0, 4).cast(pl.Int32).alias("season_year")
)
sy_mean = team_view["season_year"].mean()
sy_std  = team_view["season_year"].std(ddof=1)
team_view = team_view.with_columns(
    ((pl.col("season_year") - sy_mean) / sy_std).alias("season_zscore")
)

log.info(f"ts_pct_zscore : mean={team_view['ts_pct_zscore'].mean():.4f}, std={team_view['ts_pct_zscore'].std():.4f}")
log.info(f"pace_zscore   : mean={team_view['pace_zscore'].mean():.4f}, std={team_view['pace_zscore'].std():.4f}")
log.info(f"season_zscore : [{team_view['season_zscore'].min():.2f}, {team_view['season_zscore'].max():.2f}]")

### 8. Ricostruzione vista partita

#### 8.1 Win rate stagione precedente per team

In [ ]:
# Win rate per (team_id, season_year) — aggregato su tutto il campionato
win_rate_season = (
    team_view
    .group_by(["team_id", "season_year"])
    .agg((pl.col("wl") == "W").mean().alias("win_rate_season"))
)

# Lookup table: per ogni (team, anno S) conserva il win_rate
# che verrà joinato sulla stagione successiva (anno S+1)
win_rate_prev = (
    win_rate_season
    .with_columns((pl.col("season_year") + 1).alias("lookup_year"))
    .rename({"win_rate_season": "win_rate_prev_season"})
    .select(["team_id", "lookup_year", "win_rate_prev_season"])
)

team_view = team_view.join(
    win_rate_prev,
    left_on=["team_id", "season_year"],
    right_on=["team_id", "lookup_year"],
    how="left",
)

null_wrps = team_view["win_rate_prev_season"].null_count()
log.info(
    f"win_rate_prev_season: null={null_wrps:,} "
    f"(team alla prima stagione nel dataset o espansi dopo il 2000-01)"
)

#### 8.2 Join Home / Away → vista a livello di partita

Ogni riga del `game_view` rappresenta una singola partita con le feature di entrambe le
squadre affiancate (`home_*` e `away_*`). `season_zscore` è una feature di livello partita
(uguale per entrambi i team) e viene portata una sola volta dal lato home.

In [ ]:
FEATURE_COLS = [
    f"net_rating_rolling_{WINDOW_N}",
    f"avg_pts_last_{WINDOW_N}",
    f"avg_ts_pct_last_{WINDOW_N}",
    f"avg_pace_last_{WINDOW_N}",
    f"winrate_last_{WINDOW_N}",
    "streak",
    "rest_days",
    "is_back_to_back",
    "fg_pct",
    "fg3_pct",
    "ft_pct",
    "ts_pct_zscore",
    "pace_zscore",
    "win_rate_prev_season",
]

TARGET_RAW   = ["wl", "pts"]
META_COLS    = ["game_id", "game_date", "season", "season_zscore"]
TEAM_ID_COLS = ["team_id", "team_abbreviation"]

home_df = (
    team_view.filter(pl.col("home_away") == "Home")
    .select(META_COLS + TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS)
    .rename({c: f"home_{c}" for c in TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS})
)

away_df = (
    team_view.filter(pl.col("home_away") == "Away")
    .select(["game_id"] + TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS)
    .rename({c: f"away_{c}" for c in TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS})
)

game_view = home_df.join(away_df, on="game_id", how="inner")

log.info(f"Game view: {game_view.shape[0]:,} rows × {game_view.shape[1]} columns")
log.info(f"Colonne home_* (prime 5): {[c for c in game_view.columns if c.startswith('home_')][:5]}")
game_view.head(2)

### 9. Feature differenziali + target

Per ogni coppia `home_X` / `away_X` viene calcolato `X_diff = home_X − away_X`.
I differenziali catturano il *vantaggio relativo* della squadra di casa su quella ospite.

**Target**:
- **`home_win`** (Int8, binario): 1 se la squadra di casa ha vinto.
- **`point_differential`** (Float64): `home_pts − away_pts`.

In [ ]:
NUMERIC_FEAT = [c for c in FEATURE_COLS if c != "is_back_to_back"]
BOOL_FEAT    = ["is_back_to_back"]

diff_exprs = (
    [
        (pl.col(f"home_{c}") - pl.col(f"away_{c}")).alias(f"{c}_diff")
        for c in NUMERIC_FEAT
    ]
    + [
        (
            pl.col(f"home_{c}").cast(pl.Int8) - pl.col(f"away_{c}").cast(pl.Int8)
        ).alias(f"{c}_diff")
        for c in BOOL_FEAT
    ]
)

game_view = game_view.with_columns(
    diff_exprs
    + [
        (pl.col("home_wl") == "W").cast(pl.Int8).alias("home_win"),
        (
            pl.col("home_pts").cast(pl.Float64) - pl.col("away_pts").cast(pl.Float64)
        ).alias("point_differential"),
    ]
)

home_win_rate = float(game_view["home_win"].mean())
log.info(f"Feature differenziali calcolate : {len(diff_exprs)}")
log.info(
    f"Target home_win: {game_view['home_win'].sum():,} vittorie home / "
    f"{game_view.height:,} partite ({home_win_rate:.3f})"
)
log.info(
    f"Target point_differential: "
    f"mean={game_view['point_differential'].mean():.2f}, "
    f"std={game_view['point_differential'].std():.2f}"
)

### 10. Flag stagione anomala — COVID Bubble (2019-20)

La stagione 2019-20 è stata completata in una bolla (Orlando, FL) senza pubblico e senza
trasferte tradizionali. Questo elimina de facto l'home advantage e distorce `pace`, `ts_pct`
e i pattern di `rest_days` rispetto alle stagioni regolari (H4 dall'EDA Bivariata).

**Decisione documentata**: si mantiene la stagione nel dataset aggiungendo il flag
`is_covid_bubble = 1`. L'esclusione della stagione sarà testata durante la modellazione
(confronto AUC / RMSE con `is_covid_bubble` come feature vs. esclusione completa).

In [ ]:
game_view = game_view.with_columns(
    (pl.col("season") == "2019-20").cast(pl.Int8).alias("is_covid_bubble")
)

n_bubble = int(game_view["is_covid_bubble"].sum())
log.info(f"is_covid_bubble: {n_bubble:,} partite flaggate (stagione 2019-20)")
log.info(
    "Decisione: MANTIENI con flag. "
    "Esclusione da testare in modellazione (ipotesi H4 dall'EDA Bivariata)."
)

### 11. Filtering — MIN_GAMES_REQUIRED

Vengono scartate le partite in cui almeno uno dei due team ha meno di `MIN_GAMES_REQUIRED`
partite precedenti nel dataset. Queste righe hanno `null` nei rolling features (per via di
`min_periods=MIN_GAMES_REQUIRED`). Si usa `net_rating_rolling_N` come colonna sentinella.

In [ ]:
SENTINEL = f"net_rating_rolling_{WINDOW_N}"

n_before = game_view.height
game_view_filtered = game_view.filter(
    pl.col(f"home_{SENTINEL}").is_not_null()
    & pl.col(f"away_{SENTINEL}").is_not_null()
)
n_after   = game_view_filtered.height
n_dropped = n_before - n_after

log.info(f"Righe prima del filtering  : {n_before:,}")
log.info(
    f"Righe scartate (< {MIN_GAMES_REQUIRED} partite precedenti "
    f"per almeno un team): {n_dropped:,} ({n_dropped / n_before:.1%})"
)
log.info(f"Righe dopo il filtering    : {n_after:,}")

### 12. Train / Val / Test split temporale

Split per stagione (**nessuno shuffle**) per rispettare la causalità temporale.
Nessuna partita di una stagione successiva può comparire in un set precedente.

| Set | Stagioni | Motivazione |
|-----|---------|-------------|
| **Train** | 2000-01 → 2018-19 | ~19 stagioni per addestrare il modello |
| **Val**   | 2019-20 → 2021-22 | 3 stagioni recenti per tuning iperparametri |
| **Test**  | 2022-23 → 2025-26 | 4 stagioni per valutazione finale hold-out |

In [ ]:
TRAIN_END  = "2018-19"
VAL_START  = "2019-20"
VAL_END    = "2021-22"
TEST_START = "2022-23"

train_df = game_view_filtered.filter(pl.col("season") <= TRAIN_END)
val_df   = game_view_filtered.filter(
    (pl.col("season") >= VAL_START) & (pl.col("season") <= VAL_END)
)
test_df  = game_view_filtered.filter(pl.col("season") >= TEST_START)

# Sanity checks — nessun overlap temporale
assert train_df.height + val_df.height + test_df.height == game_view_filtered.height,     "Split non esaustivo — verificare le boundary stagionali."
assert str(train_df["season"].max()) < str(val_df["season"].min()),     "Overlap Train/Val rilevato!"
assert str(val_df["season"].max()) < str(test_df["season"].min()),     "Overlap Val/Test rilevato!"

total = game_view_filtered.height
for name, split in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    pct = split.height / total
    log.info(
        f"{name}: {split.height:,} partite ({pct:.1%}) | "
        f"stagioni {split['season'].min()} – {split['season'].max()}"
    )

### 13. Salvataggio

In [ ]:
game_view_final = game_view_filtered.with_columns(
    pl.when(pl.col("season") <= TRAIN_END).then(pl.lit("train"))
      .when(pl.col("season") <= VAL_END).then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("split")
)

game_view_final.write_csv(OUT_FILE)

log.info(f"File salvato  : {OUT_FILE}")
log.info(f"Shape finale  : {game_view_final.height:,} rows × {game_view_final.width} columns")

counts = {s: game_view_final.filter(pl.col("split") == s).height for s in ["train", "val", "test"]}
log.info(f"Split → train: {counts['train']:,} | val: {counts['val']:,} | test: {counts['test']:,}")

log.info(f"\nColonne nel dataset finale ({game_view_final.width}):")
for i, c in enumerate(game_view_final.columns, 1):
    log.info(f"  {i:2d}. {c}")